# Package C: Population data cleaning and country matching

**Team 2: Smart Waste Analytics & Demographics** (owner: Islam Md Shajedul)

This notebook cleans the UN World Population Prospects 2024 data and checks it against the cleaned waste data from package B.

**Steps**
1. Load the raw WPP 2024 total population file
2. Keep only countries (not regions or world totals) and only the Medium variant, 1950–2100
3. Add the yearly population growth rate
4. Flag projection years (after 2023)
5. Build a country to sub-region and region lookup
6. Check that every country in B's waste files has a population match
7. Save two clean CSVs to `processed/`

Raw files are never edited. Everything is done in code.

## 1. Setup and file paths

In [ ]:
from pathlib import Path

import pandas as pd

# Works whether the notebook is run from team2_waste_data/scripts/,
# from team2_waste_data/, or from the repository root.
HERE = Path.cwd()
if HERE.name == "scripts":
    DATA_DIR = HERE.parent
elif (HERE / "team2_waste_data").exists():
    DATA_DIR = HERE / "team2_waste_data"
else:
    DATA_DIR = HERE

RAW_POP = DATA_DIR / "raw" / "unpop"
PROCESSED = DATA_DIR / "processed"
PROCESSED.mkdir(exist_ok=True)

LAST_OBSERVED_YEAR = 2023   # years after this are UN projections, not observed data
FIRST_YEAR, LAST_YEAR = 1950, 2100
WORLD_ID = 900              # M49 code of "World", the top of the location tree

print("Raw population folder:", RAW_POP.resolve())
print("Output folder:        ", PROCESSED.resolve())

## 2. Load the raw population file

The file is UTF-8 with a byte-order mark, so it is read with `encoding="utf-8-sig"` (known trap 6 in the data README).
If only the `.gz` version is present, pandas reads it directly.

In [ ]:
def find_raw_file(name):
    """Return the path of a raw file, accepting either the .csv or the .csv.gz version."""
    for candidate in (RAW_POP / name, RAW_POP / (name + ".gz")):
        if candidate.exists():
            return candidate
    raise FileNotFoundError(f"{name} not found in {RAW_POP}. Run scripts/download_data.py first.")


pop_raw = pd.read_csv(find_raw_file("WPP2024_TotalPopulationBySex.csv"),
                      encoding="utf-8-sig", low_memory=False)

print("Raw shape:", pop_raw.shape)   # expected (720210, 18) for the full download
pop_raw.head()

## 3. Keep countries only, Medium variant, 1950–2100

- The file mixes countries with aggregates (world, regions, income groups). Countries are the rows with `LocTypeID == 4`.
- We filter on the variant **name** `"Medium"`, not a number, because API variant IDs and WPP `VarID` values differ (known trap 3).
- Population values are in **thousands** of people.

In [ ]:
is_country = pop_raw["LocTypeID"] == 4
is_medium = pop_raw["Variant"] == "Medium"
in_range = pop_raw["Time"].between(FIRST_YEAR, LAST_YEAR)

countries = pop_raw[is_country & is_medium & in_range].copy()

print("Rows kept:", len(countries))
print("Countries:", countries["LocID"].nunique(), "(README expects 237)")
print("Years:", countries["Time"].min(), "to", countries["Time"].max())

## 4. Rename and tidy the columns

In [ ]:
pop = (
    countries
    .rename(columns={
        "LocID": "CountryID",          # UN M49 code, same key as UNSD CountryID
        "ISO3_code": "ISO3",
        "Location": "Country",
        "Time": "Year",
        "PopTotal": "Population_1000",  # thousands of people
        "PopDensity": "Pop_Density_per_km2",
    })
    [["CountryID", "ISO3", "Country", "Year", "Population_1000", "Pop_Density_per_km2"]]
    .sort_values(["CountryID", "Year"])
    .reset_index(drop=True)
)

pop["CountryID"] = pop["CountryID"].astype("Int64")
pop["Year"] = pop["Year"].astype(int)
pop["Country"] = pop["Country"].str.strip()

# Each country must have exactly one row per year
assert not pop.duplicated(["CountryID", "Year"]).any(), "Duplicate country-year rows found"
assert pop["Population_1000"].notna().all(), "Missing population values found"
pop.head()

## 5. Add the yearly growth rate

Growth rate (%) = (population this year − population last year) / population last year × 100.

It is calculated separately for each country (`groupby`), so one country's numbers never mix with the next.
The first year (1950) has no previous year, so its growth rate is empty (NaN).

In [ ]:
pop["Growth_Rate_Pct"] = pop.groupby("CountryID")["Population_1000"].pct_change() * 100

pop[["Country", "Year", "Population_1000", "Growth_Rate_Pct"]].head()

## 6. Flag projection years

WPP values after 2023 are UN projections (Medium variant), not measured data.
`Is_Projection` lets D, E and F treat them differently, for example by drawing them as a dashed line.

In [ ]:
pop["Is_Projection"] = pop["Year"] > LAST_OBSERVED_YEAR

pop["Is_Projection"].value_counts().rename({False: "observed (<= 2023)", True: "projection (> 2023)"})

## 7. Build the country → sub-region → region lookup

Every location in the WPP file has a `ParentID`. For a country, the parent is its sub-region
(for example Kenya → Eastern Africa). We walk up the chain of parents until we reach World:

- **SubRegion** = the country's direct parent (22 UN sub-regions, e.g. Eastern Africa, South America)
- **Region** = the top geographic region (5 continents: Africa, Americas, Asia, Europe, Oceania)

The region names come straight from the UN file, so we don't type any of them by hand.

In [ ]:
locations = (
    pop_raw[["LocID", "Location", "LocTypeName", "ParentID"]]
    .drop_duplicates("LocID")
    .set_index("LocID")
)


# Data fix: USA, Canada, Bermuda, Greenland and Saint Pierre and Miquelon point to parent 918
# ("Northern America" sub-region), but WPP 2024 has no row with LocID 918. The same area exists
# as LocID 905 ("Northern America", under Americas), so we use 905 for these countries.
PARENT_FIXES = {918: 905}


def ancestors(loc_id):
    """Return the list of parent location IDs of loc_id, from nearest to furthest, stopping before World."""
    chain = []
    parent = locations.at[loc_id, "ParentID"]
    while pd.notna(parent):
        parent = PARENT_FIXES.get(int(parent), int(parent))
        if parent == WORLD_ID or parent not in locations.index:
            break
        chain.append(parent)
        parent = locations.at[parent, "ParentID"]
    return chain


rows = []
for country_id, iso3, name in pop[["CountryID", "ISO3", "Country"]].drop_duplicates().itertuples(index=False):
    chain = ancestors(int(country_id))
    rows.append({
        "CountryID": country_id,
        "ISO3": iso3,
        "Country": name,
        "SubRegion": locations.at[chain[0], "Location"] if chain else pd.NA,
        "Region": locations.at[chain[-1], "Location"] if chain else pd.NA,
    })

region_lookup = pd.DataFrame(rows).sort_values("CountryID").reset_index(drop=True)
region_lookup["CountryID"] = region_lookup["CountryID"].astype("Int64")

missing_region = region_lookup[region_lookup["Region"].isna()]
print("Countries without a region:", len(missing_region))
assert missing_region.empty, missing_region
print()
print("Countries per region:")
print(region_lookup["Region"].value_counts().to_string())

## 8. Check every waste country has a population match

The key is the M49 code (`CountryID` in B's files, `LocID` in WPP). Names are not used because they differ in punctuation and formal names (known trap 8).

In [ ]:
WASTE_FILES = [
    "municipal_waste_collected_clean.csv",
    "municipal_waste_recycled_pct_clean.csv",
    "municipal_waste_treatment_latest_clean.csv",
]

population_ids = set(pop["CountryID"].dropna().astype(int))
unmatched = []

for file_name in WASTE_FILES:
    path = PROCESSED / file_name
    if not path.exists():
        print(f"SKIPPED {file_name}: not found in processed/ (run package B's notebook first)")
        continue
    waste = pd.read_csv(path)
    waste_countries = waste[["CountryID", "Country"]].drop_duplicates("CountryID")
    misses = waste_countries[~waste_countries["CountryID"].astype(int).isin(population_ids)]
    print(f"{file_name}: {waste_countries['CountryID'].nunique()} countries, {len(misses)} without population data")
    for cid, name in misses.itertuples(index=False):
        unmatched.append({"CountryID": cid, "Country": name, "Waste_File": file_name})

unmatched = pd.DataFrame(unmatched, columns=["CountryID", "Country", "Waste_File"])
unmatched

**Result on the real data:** only 2 waste countries have no population match:
- **Sudan (Former), 736**: Sudan before South Sudan split off in 2011. WPP only has today's Sudan (729) and South Sudan (728).
- **Kosovo\*, 383**: WPP 2024 has no separate row for Kosovo.

Package A should drop these from the join or explain them in the report. All other countries match by M49 code.

## 9. Save the clean files

In [ ]:
pop = pop[["CountryID", "ISO3", "Country", "Year", "Population_1000",
           "Pop_Density_per_km2", "Growth_Rate_Pct", "Is_Projection"]]

pop.to_csv(PROCESSED / "population_clean.csv", index=False)
region_lookup.to_csv(PROCESSED / "country_region_lookup.csv", index=False)
unmatched.to_csv(PROCESSED / "population_unmatched_countries.csv", index=False)

print("population_clean.csv:          ", pop.shape)
print("country_region_lookup.csv:     ", region_lookup.shape)
print("population_unmatched_countries:", unmatched.shape)

## Output summary

| File | One row per | Columns |
|---|---|---|
| `population_clean.csv` | country and year (1950–2100) | CountryID, ISO3, Country, Year, Population_1000, Pop_Density_per_km2, Growth_Rate_Pct, Is_Projection |
| `country_region_lookup.csv` | country | CountryID, ISO3, Country, SubRegion, Region |
| `population_unmatched_countries.csv` | waste country with no population match | CountryID, Country, Waste_File |

**Join key for package A:** `CountryID` (UN M49) + `Year`.
Waste per person (kg) = `Waste_Collected_1000t × 1,000,000 / (Population_1000 × 1,000)` = `Waste_Collected_1000t / Population_1000 × 1000`.